In [ ]:
import os
import re
import sys

import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform

sys.path.insert(0, "../../packages/ProMERGE_utils")
from promerge_utils import ppi_from_raw_ppi

In [ ]:
os.makedirs("../../result/Hela/input/", exist_ok=True)

### Gene names

In [ ]:
hgnc = pd.read_csv(
    "../../data/HGNC/hgnc_complete_set.txt",
    sep="\t",
    dtype=str,
    usecols=["symbol", "prev_symbol", "uniprot_ids"],
)


def unique_lookup(column):
    """
    Map each HGNC column to its approved symbol
    keeping only those map to a single symbol.
    """
    pairs = hgnc[["symbol", column]].dropna()
    pairs = pairs.assign(key=pairs[column].str.split("|"))
    pairs = pairs.explode("key")
    n_symbol = pairs.groupby("key")["symbol"].transform("nunique")
    pairs = pairs[n_symbol == 1]
    return dict(zip(pairs["key"], pairs["symbol"]))


APPROVED = set(hgnc["symbol"])
PREVIOUS = unique_lookup("prev_symbol")
UNIPROT = unique_lookup("uniprot_ids")


def harmonise(name):
    """
    Each id is tried as an approved or previous symbol,
    then as a UniProt accession;
    the first name is returned when nothing matches.
    """
    if not isinstance(name, str):
        return None
    tokens = [x.strip() for x in name.split("|") if x.strip()]
    for token in tokens:
        # upper-cased open reading frames, C1ORF123 -> C1orf123
        token = re.sub(r"^(C\w+)ORF(\d+)$", r"\1orf\2", token)
        if token in APPROVED:
            return token
        for lookup in [PREVIOUS, UNIPROT]:
            if token in lookup:
                return lookup[token]
    return tokens[0] if tokens else None

### SEC-MS (Heusel et al. 2020, Table ST1)

In [ ]:
sec = pd.read_excel(
    "../../data/SECMS/1-s2.0-S2405471220300016-mmc2.xlsx",
    sheet_name="ST1",
)

# UniProt accession first, the table's own gene name as fallback
first_name = sec["Gene_names"].str.split().str[0]
sec.index = [
    UNIPROT.get(accession) or harmonise(name)
    for accession, name in zip(sec["uniprot_id"], first_name)
]
sec = sec[sec.index.notna()]

In [ ]:
# Replicate-mean intensity per fraction, scaled to sum 1 per protein.
sec_profile = {}
for context, phase in {
    "Interphase": "interphase",
    "Mitosis": "mitosis"
}.items():
    columns = [
        c for c in sec.columns
        if re.fullmatch(rf"F\d+\.{phase}\.intensity\.mean", c)
    ]
    x = sec[columns].copy()
    x.columns = range(1, len(columns) + 1)
    x = x[x.sum(axis=1) > 0]
    x = x.div(x.sum(axis=1), axis=0)
    x = x.groupby(x.index).mean()
    x.index.name = 0
    x.to_csv(
        f"../../result/Hela/input/SEC-{context}.data",
        sep="\t",
        index=True,
        header=True,
    )
    sec_profile[context] = x

### TPP-TR (Becher et al. 2018, Table S4)

In [ ]:
tpp = pd.read_excel(
    "../../data/TPP/mmc4.xlsx",
    sheet_name="TableS4_TPP-TR",
)

# fix gene names that Excel turned into dates
EXCEL_DATE = {
    "2017-03-01 00:00:00": "MARCHF1",
    "2017-03-02 00:00:00": "MARCHF2",
    "2017-03-05 00:00:00": "MARCHF5",
    "2017-03-06 00:00:00": "MARCHF6",
    "2017-09-15 00:00:00": "SELENOF",
    "2017-09-10 00:00:00": "SEPTIN10",
    "2017-09-11 00:00:00": "SEPTIN11",
    "2017-09-02 00:00:00": "SEPTIN2",
    "2017-09-06 00:00:00": "SEPTIN6",
    "2017-09-07 00:00:00": "SEPTIN7",
    "2017-09-08 00:00:00": "SEPTIN8",
    "2017-09-09 00:00:00": "SEPTIN9",
}
name = tpp["gene_name"].astype(str).replace(EXCEL_DATE)
tpp["gene_name"] = name.map({n: harmonise(n) for n in name.unique()})
tpp = tpp[tpp["gene_name"].notna()]
tpp = tpp[tpp["found.in.reps"] >= 2]

# 37 C is the reference temperature and constant across proteins
temperature = [
    c for c in tpp.columns
    if c.endswith("_mean.fc") and c != "T37_mean.fc"
]

In [ ]:
tpp_profile = {}
for context, phase in {
    "Interphase": "G1_S",
    "Mitosis": "M"
}.items():
    x = tpp[tpp["cell.cycle"] == phase]
    x = x.groupby("gene_name")[temperature].mean().dropna()

    # identical curves are one protein group reported under several names
    # keep one name, preferring an approved HGNC symbol
    order = sorted(x.index, key=lambda g: (g not in APPROVED, g))
    x = x.loc[order]
    n_twin = int(x.duplicated().sum())
    x = x[~x.duplicated()].sort_index()

    x.columns = range(1, len(temperature) + 1)
    x.index.name = 0
    x.to_csv(
        f"../../result/Hela/input/TPP-{context}.data",
        sep="\t", index=True, header=True
    )
    tpp_profile[context] = x

### kNN graphs

In [ ]:
def similarity(modality, x):
    """
    Pearson r for SEC elution profiles
    1 / (1 + Euclidean distance) for TPP melting curves
    """
    values = x.to_numpy(dtype=float)
    if modality == "SEC":
        sim = np.corrcoef(values)
    else:
        sim = 1.0 / (1.0 + squareform(pdist(values)))
    return pd.DataFrame(sim, index=x.index, columns=x.index)


def knn_edges(sim, k=10):
    """
    Symmetric kNN graph from a gene-gene similarity matrix
    """
    genes = sim.index.to_numpy()
    score = np.nan_to_num(sim.to_numpy(dtype=float), nan=-np.inf)
    np.fill_diagonal(score, -np.inf)
    partner = np.argpartition(-score, k, axis=1)[:, :k]
    row = np.repeat(np.arange(len(genes)), k)
    col = partner.ravel()
    value = score[row, col]
    found = np.isfinite(value)
    a = genes[row[found]]
    b = genes[col[found]]
    edges = pd.DataFrame({
        "GeneA": np.where(a < b, a, b),
        "GeneB": np.where(a < b, b, a),
        "Score": value[found],
    })
    edges = edges.drop_duplicates(["GeneA", "GeneB"])
    return edges.sort_values("Score", ascending=False)

In [ ]:
profile = {"SEC": sec_profile, "TPP": tpp_profile}
for modality, by_context in profile.items():
    for context, x in by_context.items():
        edges = knn_edges(similarity(modality, x))
        edges.to_csv(
            f"../../result/Hela/input/{modality}-{context}.ppi",
            sep="\t", index=False, header=True
        )
        ppi_from_raw_ppi(
            f"../../result/Hela/input/{modality}-{context}.ppi",
            f"../../result/Hela/ppi/FromRaw_{modality}_{context}.ppi",
        )